In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Paths
artifacts_dir = Path("data/artifacts")
features_dir = artifacts_dir / "features"

# Load processed features
X_train = pd.read_parquet(
    features_dir / "05_X_train.parquet"
)

X_validation = pd.read_parquet(
    features_dir / "05_X_validation.parquet"
)

X_test = pd.read_parquet(
    features_dir / "05_X_test.parquet"
)

# Load targets
y_train = pd.read_parquet(
    features_dir / "05_y_train.parquet"
).squeeze()

y_validation = pd.read_parquet(
    features_dir / "05_y_validation.parquet"
).squeeze()

y_test = pd.read_parquet(
    features_dir / "05_y_test.parquet"
).squeeze()

print("Data loaded successfully.")

Data loaded successfully.


In [2]:
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

print("\nTarget distribution - Train:")
print(y_train.value_counts())

print("\nTarget distribution - Validation:")
print(y_validation.value_counts())

print("\nTarget distribution - Test:")
print(y_test.value_counts())

X_train: (67529, 39)
X_validation: (14470, 39)
X_test: (14471, 39)

y_train: (67529,)
y_validation: (14470,)
y_test: (14471,)

Target distribution - Train:
is_delayed
0    61433
1     6096
Name: count, dtype: int64

Target distribution - Validation:
is_delayed
0    13697
1      773
Name: count, dtype: int64

Target distribution - Test:
is_delayed
0    13514
1      957
Name: count, dtype: int64


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train, y_train)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [4]:
y_val_pred = baseline_model.predict(X_validation)
y_val_proba = baseline_model.predict_proba(X_validation)[:, 1]

print("Validation Results")
print("=" * 40)

print("Accuracy :", accuracy_score(y_validation, y_val_pred))
print("Precision:", precision_score(y_validation, y_val_pred, zero_division=0))
print("Recall   :", recall_score(y_validation, y_val_pred, zero_division=0))
print("F1 Score :", f1_score(y_validation, y_val_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_validation, y_val_proba))

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_val_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, y_val_pred))

Validation Results
Accuracy : 0.9464409122322046
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
ROC-AUC  : 0.7727729256961398

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     13697
           1       0.00      0.00      0.00       773

    accuracy                           0.95     14470
   macro avg       0.47      0.50      0.49     14470
weighted avg       0.90      0.95      0.92     14470


Confusion Matrix:
[[13695     2]
 [  773     0]]


In [5]:
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

balanced_model.fit(X_train, y_train)

print("Balanced Logistic Regression trained successfully.")

Balanced Logistic Regression trained successfully.


In [6]:
y_val_pred_balanced = balanced_model.predict(X_validation)
y_val_proba_balanced = balanced_model.predict_proba(
    X_validation
)[:, 1]

print("Balanced Logistic Regression - Validation Results")
print("=" * 50)

print("Accuracy :",
      accuracy_score(y_validation, y_val_pred_balanced))

print("Precision:",
      precision_score(
          y_validation,
          y_val_pred_balanced,
          zero_division=0
      ))

print("Recall   :",
      recall_score(
          y_validation,
          y_val_pred_balanced,
          zero_division=0
      ))

print("F1 Score :",
      f1_score(
          y_validation,
          y_val_pred_balanced,
          zero_division=0
      ))

print("ROC-AUC  :",
      roc_auc_score(
          y_validation,
          y_val_proba_balanced
      ))

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_val_pred_balanced,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        y_val_pred_balanced
    )
)

Balanced Logistic Regression - Validation Results
Accuracy : 0.6848652384243262
Precision: 0.1111111111111111
Recall   : 0.6998706338939198
F1 Score : 0.19177596596951435
ROC-AUC  : 0.7691726434462518

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.68      0.80     13697
           1       0.11      0.70      0.19       773

    accuracy                           0.68     14470
   macro avg       0.54      0.69      0.50     14470
weighted avg       0.93      0.68      0.77     14470


Confusion Matrix:
[[9369 4328]
 [ 232  541]]


In [7]:
threshold_results = []

thresholds = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

for threshold in thresholds:

    y_pred = (
        y_val_proba_balanced >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "accuracy": accuracy_score(
            y_validation,
            y_pred
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

 threshold  precision   recall       f1  accuracy
      0.10   0.054139 1.000000 0.102717  0.066690
      0.15   0.055452 0.998706 0.105070  0.091154
      0.20   0.057666 0.997413 0.109029  0.129164
      0.25   0.060513 0.989651 0.114051  0.178645
      0.30   0.065322 0.984476 0.122515  0.246648
      0.35   0.072450 0.963777 0.134768  0.338908
      0.40   0.083596 0.927555 0.153369  0.452937
      0.45   0.095011 0.825356 0.170406  0.570698
      0.50   0.111111 0.699871 0.191776  0.684865


In [8]:
threshold_results_fine = []

thresholds_fine = np.arange(0.50, 0.71, 0.01)

for threshold in thresholds_fine:
    y_pred = (
        y_val_proba_balanced >= threshold
    ).astype(int)

    threshold_results_fine.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_validation, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_validation, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_validation, y_pred, zero_division=0
        ),
        "accuracy": accuracy_score(
            y_validation, y_pred
        )
    })

threshold_fine_df = pd.DataFrame(threshold_results_fine)

print(
    threshold_fine_df.to_string(index=False)
)

best_threshold_row = threshold_fine_df.loc[
    threshold_fine_df["f1"].idxmax()
]

print("\nBest threshold by F1:")
print(best_threshold_row)

 threshold  precision   recall       f1  accuracy
      0.50   0.111111 0.699871 0.191776  0.684865
      0.51   0.113212 0.666235 0.193536  0.703386
      0.52   0.117964 0.650712 0.199722  0.721424
      0.53   0.122143 0.622251 0.204203  0.740912
      0.54   0.127322 0.602846 0.210241  0.758051
      0.55   0.130243 0.574386 0.212339  0.772357
      0.56   0.133778 0.544631 0.214796  0.787284
      0.57   0.140055 0.526520 0.221256  0.802004
      0.58   0.142911 0.494179 0.221706  0.814651
      0.59   0.146940 0.472186 0.224133  0.825363
      0.60   0.153057 0.463131 0.230077  0.834416
      0.61   0.156009 0.445019 0.231028  0.841742
      0.62   0.159434 0.423027 0.231586  0.850035
      0.63   0.162246 0.403622 0.231454  0.856807
      0.64   0.167513 0.384217 0.233307  0.865100
      0.65   0.173432 0.364812 0.235098  0.873186
      0.66   0.183165 0.351876 0.240921  0.881548
      0.67   0.189528 0.332471 0.241428  0.888390
      0.68   0.190204 0.301423 0.233233  0.894126


In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_val_proba_rf = rf_model.predict_proba(
    X_validation
)[:, 1]

y_val_pred_rf = (
    y_val_proba_rf >= 0.5
).astype(int)

print("Random Forest - Validation Results")
print("-----------------------------------")

print(
    "Accuracy:",
    accuracy_score(y_validation, y_val_pred_rf)
)

print(
    "Precision:",
    precision_score(
        y_validation,
        y_val_pred_rf,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_validation,
        y_val_pred_rf,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_validation,
        y_val_pred_rf,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_validation,
        y_val_proba_rf
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        y_val_pred_rf
    )
)

Random Forest - Validation Results
-----------------------------------
Accuracy: 0.9465791292328957
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.7300111798685673

Confusion Matrix:
[[13697     0]
 [  773     0]]


In [10]:
rf_threshold_results = []

rf_thresholds = np.arange(0.10, 0.71, 0.01)

for threshold in rf_thresholds:
    y_pred = (
        y_val_proba_rf >= threshold
    ).astype(int)

    rf_threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "accuracy": accuracy_score(
            y_validation,
            y_pred
        )
    })

rf_threshold_df = pd.DataFrame(
    rf_threshold_results
)

print(
    rf_threshold_df.to_string(index=False)
)

best_rf_threshold_row = rf_threshold_df.loc[
    rf_threshold_df["f1"].idxmax()
]

print("\nBest Random Forest threshold by F1:")
print(best_rf_threshold_row)

 threshold  precision   recall       f1  accuracy
      0.10   0.144031 0.385511 0.209711  0.844782
      0.11   0.153890 0.345408 0.212919  0.863580
      0.12   0.160579 0.301423 0.209532  0.878507
      0.13   0.171119 0.265201 0.208016  0.892122
      0.14   0.183920 0.236740 0.207014  0.903110
      0.15   0.192532 0.213454 0.202454  0.910159
      0.16   0.186486 0.178525 0.182419  0.914513
      0.17   0.202552 0.164295 0.181429  0.920802
      0.18   0.200758 0.137128 0.162952  0.924741
      0.19   0.208243 0.124191 0.155592  0.927989
      0.20   0.225564 0.116429 0.153584  0.931444
      0.21   0.226087 0.100906 0.139535  0.933518
      0.22   0.228758 0.090556 0.129750  0.935107
      0.23   0.232210 0.080207 0.119231  0.936697
      0.24   0.243243 0.069858 0.108543  0.938701
      0.25   0.258242 0.060802 0.098429  0.940498
      0.26   0.256757 0.049159 0.082519  0.941603
      0.27   0.266129 0.042691 0.073579  0.942571
      0.28   0.290909 0.041397 0.072480  0.943400


In [11]:
import joblib
from pathlib import Path

model_dir = Path("data/artifacts/model")
model_dir.mkdir(parents=True, exist_ok=True)

# Save selected model
joblib.dump(
    balanced_model,
    model_dir / "06_logistic_regression_balanced.joblib"
)

# Save selected threshold
selected_threshold = 0.67

threshold_info = pd.DataFrame({
    "model": ["Balanced Logistic Regression"],
    "threshold": [selected_threshold],
    "precision": [0.189528],
    "recall": [0.332471],
    "f1": [0.241428],
    "accuracy": [0.888390],
    "roc_auc": [0.769173]
})

threshold_info.to_csv(
    model_dir / "06_selected_model_validation.csv",
    index=False
)

print("Model and validation results saved successfully.")

print("\nSaved files:")
for file in sorted(model_dir.iterdir()):
    print(file)

Model and validation results saved successfully.

Saved files:
data\artifacts\model\06_logistic_regression_balanced.joblib
data\artifacts\model\06_selected_model_validation.csv


In [12]:
# Final evaluation on Test set

selected_threshold = 0.67

# Predict probabilities
y_test_proba = balanced_model.predict_proba(X_test)[:, 1]

# Apply selected threshold
y_test_pred = (
    y_test_proba >= selected_threshold
).astype(int)

# Calculate final metrics
test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_proba
)

print("Final Test Results")
print("------------------")
print(f"Threshold: {selected_threshold}")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_test_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0
    )
)

Final Test Results
------------------
Threshold: 0.67
Accuracy:  0.7891
Precision: 0.0941
Recall:    0.2539
F1 Score:  0.1374
ROC-AUC:   0.6825

Confusion Matrix:
[[11176  2338]
 [  714   243]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.83      0.88     13514
           1       0.09      0.25      0.14       957

    accuracy                           0.79     14471
   macro avg       0.52      0.54      0.51     14471
weighted avg       0.88      0.79      0.83     14471



In [13]:
# Save final Test results

model_dir = Path("data/artifacts/model")
model_dir.mkdir(parents=True, exist_ok=True)

# 1. Test metrics
test_metrics = pd.DataFrame({
    "metric": [
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    "value": [
        selected_threshold,
        test_accuracy,
        test_precision,
        test_recall,
        test_f1,
        test_roc_auc
    ]
})

test_metrics.to_csv(
    model_dir / "06_test_metrics.csv",
    index=False
)

# 2. Confusion matrix
cm = confusion_matrix(
    y_test,
    y_test_pred
)

confusion_df = pd.DataFrame(
    cm,
    index=["actual_0", "actual_1"],
    columns=["predicted_0", "predicted_1"]
)

confusion_df.to_csv(
    model_dir / "06_confusion_matrix.csv"
)

# 3. Classification report
report = classification_report(
    y_test,
    y_test_pred,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    model_dir / "06_classification_report.csv"
)

print("Final Test artifacts saved successfully.")

print("\nSaved files:")
for file in sorted(model_dir.iterdir()):
    print(file)

Final Test artifacts saved successfully.

Saved files:
data\artifacts\model\06_classification_report.csv
data\artifacts\model\06_confusion_matrix.csv
data\artifacts\model\06_logistic_regression_balanced.joblib
data\artifacts\model\06_selected_model_validation.csv
data\artifacts\model\06_test_metrics.csv
